# **Installation of Basic Libraries**

In [2]:
!pip install -U peft bitsandbytes transformers accelerate trl PyMuPDF datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 123.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 76.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 110.5 MB/s eta 0:00:0000:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: accelerate
    Found existing installatio

# **Creating Datasets from Books**

In [3]:
import fitz
def extract_text_from_pdf(file_path:str):
    text_blocks:list=[]
    with fitz.open(file_path) as doc:
        for page in doc:
            text=page.get_text("text").strip()
            if text:
                text_blocks.append(text)
    return text_blocks

## **Get Text from Pdfs**

In [4]:
pdf_text_dataset=extract_text_from_pdf("/kaggle/input/datasets/shamiiikhaliq/ai-engineer/AI Engineer.pdf")

In [5]:
pdf_text_dataset[0]

'Chapter 1. Introduction to\nBuilding AI Applications with\nFoundation Models\nIf I could use only one word to describe AI post-2020, it’d be\nscale. The AI models behind applications like ChatGPT,\nGoogle’s Gemini, and Midjourney are at such a scale that\nthey’re consuming a nontrivial portion of the world’s electric-\nity, and we’re at risk of running out of publicly available in-\nternet data to train them.\nThe scaling up of AI models has two major consequences.\nFirst, AI models are becoming more powerful and capable of\nmore tasks, enabling more applications. More people and\nteams leverage AI to increase productivity, create economic\nvalue, and improve quality of life.\nSecond, training large language models (LLMs) requires data,\ncompute resources, and specialized talent that only a few or-\nganizations can afford. This has led to the emergence of model\nas a service: models developed by these few organizations are\nmade available for others to use as a service. Anyone who\nwi

In [6]:
import re
def split_paragraphs(pages):
    paragraphs = []
    for page_text in pages:
        # Split on double line breaks or long newlines
        chunks = re.split(r'\n\s*\n', page_text)
        for chunk in chunks:
            clean = chunk.strip()
            if len(clean) > 30:  # ignore too short lines
                paragraphs.append(clean)
    return paragraphs

In [7]:
clean_paragraphs_dataset=split_paragraphs(pdf_text_dataset)

In [8]:
dataset = [{"text":paragraph} for paragraph in clean_paragraphs_dataset]

In [9]:
dataset[0]

{'text': 'Chapter 1. Introduction to\nBuilding AI Applications with\nFoundation Models\nIf I could use only one word to describe AI post-2020, it’d be\nscale. The AI models behind applications like ChatGPT,\nGoogle’s Gemini, and Midjourney are at such a scale that\nthey’re consuming a nontrivial portion of the world’s electric-\nity, and we’re at risk of running out of publicly available in-\nternet data to train them.\nThe scaling up of AI models has two major consequences.\nFirst, AI models are becoming more powerful and capable of\nmore tasks, enabling more applications. More people and\nteams leverage AI to increase productivity, create economic\nvalue, and improve quality of life.\nSecond, training large language models (LLMs) requires data,\ncompute resources, and specialized talent that only a few or-\nganizations can afford. This has led to the emergence of model\nas a service: models developed by these few organizations are\nmade available for others to use as a service. Anyon

In [10]:
from datasets import Dataset
huggingface_type_dataset=Dataset.from_list(dataset)

## **preparing for Fine tuning**

In [11]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [12]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)

## **Loading the TinyLlama Model**

## **Short Demo for Full Fine tunning**

**We'll get the error of OutOfMemory Error**

In [13]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [14]:
import torch

print(torch.cuda.is_available())  # True if CUDA can be used
if torch.cuda.is_available():
    print(torch.cuda.device_count())
    print(torch.cuda.get_device_name(0))

True
2
Tesla T4


In [15]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    

In [16]:
def tokenize_fn(examples):
    tokens = tokenizer(
        examples["text"], truncation=True, padding="max_length", max_length=512
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [17]:
tokenized = huggingface_type_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

Map:   0%|          | 0/761 [00:00<?, ? examples/s]

In [18]:
model = AutoModelForCausalLM.from_pretrained(model_name,device_map="auto")

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

In [19]:
# training_args = TrainingArguments(
#     output_dir="./llama-pharma-domain",
#     num_train_epochs=2,
#     per_device_train_batch_size=2,
#     save_steps=500,
#     save_total_limit=2,
#     logging_steps=50,
#     learning_rate=2e-5,
#     fp16=True,
#     report_to="none",
# )

In [20]:
# from transformers import TrainingArguments

# help(TrainingArguments)

In [21]:
# trainer = Trainer(model=model, args=training_args, train_dataset=tokenized)
# trainer.train()

## **Lets start Transfer Learing Fine tuning**

## **Freezing the model layers & Unfreeze last 4 transformer blocks + lm_head**

In [22]:
for param in model.parameters():
    param.requires_grad = False

for name, param in model.named_parameters():
    if any(f"layers.{i}." in name for i in range(20, 24)):  # last 4 layers
        param.requires_grad = True
    if "lm_head" in name:
        param.requires_grad = True

# Verify how many parameters are trainable
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✅ Trainable: {trainable / total * 100:.2f}% of total parameters")

✅ Trainable: 13.97% of total parameters


In [29]:
from datasets import load_dataset
# final_dataset=load_dataset(huggingface_type_dataset)
huggingface_type_dataset

Dataset({
    features: ['text'],
    num_rows: 761
})

# **Step 1 — Convert your Python dataset to JSONL**

In [31]:
import json

# Convert dataset
dataset = [{"text": paragraph} for paragraph in clean_paragraphs_dataset]

# File path
file_path = "/kaggle/working/pharma_non_instruction.jsonl"

# Save as JSONL
with open(file_path, "w") as f:
    for row in dataset:
        f.write(json.dumps(row) + "\n")

print("Dataset saved successfully at:", file_path)


Dataset saved successfully at: /kaggle/working/pharma_non_instruction.jsonl


In [32]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={"train": "/kaggle/working/pharma_non_instruction.jsonl"}
)

print(dataset)


Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 761
    })
})


In [33]:
with open("pharma_non_instruction.jsonl", "w") as f:
    for row in dataset:
        f.write(json.dumps(row) + "\n")

In [35]:
tokenized=dataset["train"].map(tokenize_fn,batched=True,remove_columns=["text"])

Map:   0%|          | 0/761 [00:00<?, ? examples/s]

In [36]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


In [39]:
import os
os.environ["WANDB_DISABLED"] = "true"  # disable wandb
training_args = TrainingArguments(
    output_dir="./tinyllama-pharma-last4",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,  # slightly higher since fewer params
    fp16=True,
    logging_steps=20,
    save_steps=200,
    save_total_limit=2,
    report_to="none",
)

# -------------------------------------------------------------
# 9️⃣ Trainer setup
# -------------------------------------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=data_collator,
)

In [40]:
trainer.train()

Step,Training Loss
20,2.234432
40,2.196626
60,2.145771
80,2.138626
100,1.973994
120,1.629337
140,1.615370
160,1.601842
180,1.597029


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=192, training_loss=1.8845361868540447, metrics={'train_runtime': 196.8781, 'train_samples_per_second': 7.731, 'train_steps_per_second': 0.975, 'total_flos': 4836949550432256.0, 'train_loss': 1.8845361868540447, 'epoch': 2.0})

In [41]:
trainer.save_model("./tinyllama-pharma-last4-final")
print("\n✅ Training completed and model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Training completed and model saved!


In [42]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_path = "./tinyllama-pharma-last4-final"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [49]:
question = "What is Chapter 10. AI Engineering Architecture and User Feedback"

inputs = tokenizer(question, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=150,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id
)


response = tokenizer.decode(output[0], skip_special_tokens=True)

print(response)


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


What is Chapter 10. AI Engineering Architecture and User Feedback
AI engineering architecture and user feedback
AI Engineering
Architecture and User
Feedback
A model can be defined as a mathematical representation of an object, process or system. For example, the simplest models are called primitives that represent individual operations, such as addition, multiplication, or division.
The most common types of primitive are:
For-loops
Fortran has three built in loops, namely:
Loop over all values
Loop until a condition evaluates to false
Loop for all possible values
There’s also a specialized type of loop known as a nested loop, which allows you to access variables within your own loops:
Loops are often used in programming languages when it’s necessary to execute
